In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import multiprocessing as mp
import emcee
import matplotlib.pyplot as plt
import corner
from scipy.interpolate import interp1d
import beyond21.log_prob as NF
from emcee.backends import HDFBackend
import os
from scipy.stats import norm
from nautilus import Prior
from scipy.stats import multivariate_normal
from nautilus import Sampler

import pandas as pd
import beyond21.run_table as run_table



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
local = True       # True: run here, False: sbatch run_nautilus_mpi.sh

results_dir = ("Beyond21Naut/chains"
               if local else "/sci/labs/tomerv/omerzvikatz/Beyond21Naut/chains")
# pd.read_csv(f"{results_dir}/{run_table.FILENAME}")

In [ ]:
# Same run, but with the nautilus nested sampler
# (results go to mcmc_results/nautilus{chain_num}.h5)

# Chain number
chain_num = 0

# run mode: 'full', 'single_xPL', 'constant_xPL'
run_mode = 'constant_xPL'
fresh_run = True   # True: start a new run, False: resume from the .h5 file

# Likelihood terms to include (all True = the full, standard likelihood)
saras = False      # SARAS3 21cm spectrum -- also adds 7 foreground parameters to the fit
reion = True       # reionization: CMB optical depth + quasar xHI at z=5.9
cxb   = True       # unresolved cosmic X-ray background (Chandra, soft/med/hard)
uvlf  = True       # UV luminosity functions (Bouwens21 + Bouwens16 + Donnan24)

# Terms switched off are still computed and stored in the blob (last 3 entries:
# reion, cxb, uvlf), they are just not added to the total log-likelihood.

# 21cmFAST star-formation efficiency: fixes log10Mpivot = 10 (dropping it from
# the fit) and takes FstarII = 2 * 10**log10FstarII
sfe_21cmFAST = True

# Parameters held fixed instead of sampled, {name: value}. Any parameter can be
# fixed, including one the run mode would otherwise pin down itself (alpha_s is
# 1 under constant_xPL, say). Each fixed parameter drops out of theta, so the
# run has one dimension fewer, e.g. fixed = {'log10Mpivot': 10, 'Emin': 0.5}
# fixed = {}
fixed = {'log10FstarII':-1.25, 
         't_star':0.5,
         'alpha_1':-0.5,
         'alpha_2':-0.5,
         'alpha_escII' : 0,
         'alpha_escIII' : 0,
         'alpha_starIII' : -0.5
         }

# Store the 252-entry blob (CXB, T21, xHI, Q_ion, per-term log-likelihoods) with
# every sample. False keeps the .h5 small, but no T21/xHI histories are kept and
# the plotting routines that read blobs will not work on the chain.
save_blobs = False


In [ ]:
args = (f"{chain_num} {run_mode} {saras} {fresh_run} {reion} {cxb} {uvlf} "
        f"{save_blobs} {sfe_21cmFAST} {NF.format_fixed(fixed)}")
if local:
    os.system(f"python /home/bakerem/Beyond21/src/beyond21/run_nautilus_local.py {args}")
else:
    !sbatch run_nautilus_mpi.sh {args}

Likelihood terms: saras=False, reion=True, cxb=True, uvlf=True | save_blobs=False | sfe_21cmFAST=True | popII_only=False | fixed={'log10FstarII': -1.25, 't_star': 0.5, 'alpha_1': -0.5, 'alpha_2': -0.5, 'alpha_escII': 0.0, 'alpha_escIII': 0.0, 'alpha_starIII': -0.5}
Starting a fresh nautilus run: ndim=8, n_live=1000.
Settings recorded in /home/bakerem/Beyond21/tutorial/Beyond21Naut/chains/run_settings.csv
Starting the nautilus sampler...
Please report issues at github.com/johannesulf/nautilus.
Status    | Bounds | Ellipses | Networks | Calls    | f_live | N_eff | log Z    
